In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.scoutai_common import get_engine

# 1. Database Connection
engine = get_engine()

# 2. Fetch Data
query = "SELECT * FROM view_scout_master"
df = pd.read_sql(query, engine)

# 3. Select numeric columns and EXCLUDE irrelevant IDs
numeric_df = df.select_dtypes(include=['number'])
if 'player_id' in numeric_df.columns:
    numeric_df = numeric_df.drop(columns=['player_id'])

# 4. Clean Feature Names for Professional Display
# Replaces underscores with spaces and capitalizes each word (e.g., 'total_goals' -> 'Total Goals')
clean_columns = {col: col.replace('_', ' ').title() for col in numeric_df.columns}
numeric_df = numeric_df.rename(columns=clean_columns)

# 5. Calculate the Correlation Matrix
correlation_matrix = numeric_df.corr()

# 6. Visualization
plt.figure(figsize=(14, 12))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5,        # Adds clean borders between squares
    cbar_kws={"shrink": .8} # Scales the colorbar elegantly
)

plt.title("ScoutAI: Feature Correlation Matrix", fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=11)  # Angles the bottom labels for better reading
plt.yticks(rotation=0, fontsize=11)

plt.tight_layout()
plt.savefig('feature_correlation_matrix.png', dpi=150)
plt.show()

# 7. Target Variable Correlation Print
# Note: 'log_market_value' became 'Log Market Value' due to the cleaning above
target_col = 'Log Market Value'
if target_col in correlation_matrix.columns:
    target_corr = correlation_matrix[target_col].sort_values(ascending=False)
    print("\n--- Correlation with Market Value ---")
    print(target_corr)
else:
    print(f"[WARNING] '{target_col}' not found in correlation matrix columns.")
    print("Available columns:", list(correlation_matrix.columns))